In [1]:
import pandas as pd
from pathlib import Path

# 1. Flexible Paths Configuration
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

ELO_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"
STATS_PATH = BASE_DIR / "data" / "processed" / "fbref_group_stats.csv"
FIXTURES_PATH = BASE_DIR / "data" / "processed" / "upcoming_fixtures.csv"
OUTPUT_MATCHUPS_PATH = BASE_DIR / "data" / "processed" / "model_input_fixtures.csv"

print("[INFO] Loading generated datasets...")
df_elo = pd.read_csv(ELO_PATH)
df_stats = pd.read_csv(STATS_PATH)
df_fixtures = pd.read_csv(FIXTURES_PATH)

# 2. Bulletproof Name Standardization (Substring Matching)
# Exact matching fails due to invisible characters or weird hyphens. 
# We use partial string detection instead.
def standardize_team_names(name):
    name = str(name).strip()
    if "Bosnia" in name: return "Bosnia and Herzegovina"
    if "USA" in name or name == "US": return "United States"
    if "Korea" in name and "South" not in name and "North" not in name: return "South Korea"
    if "Iran" in name: return "Iran"
    if "Cote" in name or "Côte" in name or "Ivory" in name: return "Ivory Coast"
    if "Cabo Verde" in name: return "Cape Verde"
    if "Congo" in name and "DR" in name: return "DR Congo"
    return name

# Apply the foolproof mapping across all dataframes
for df in [df_stats, df_elo]:
    df['team'] = df['team'].apply(standardize_team_names)

for col in ['home_team', 'away_team']:
    df_fixtures[col] = df_fixtures[col].apply(standardize_team_names)

# 3. Data Blending (Creating Team Profiles)
df_profiles = pd.merge(df_elo, df_stats[['team', 'Poss', 'Gls']], on='team', how='inner')
print(f"[INFO] Built integrated profiles for {len(df_profiles)} out of 48 teams.")

# Anomaly Detection
tournament_teams = set(df_fixtures['home_team']).union(set(df_fixtures['away_team']))
missing_teams = tournament_teams - set(df_profiles['team'])

if missing_teams:
    print(f"\n[WARNING] DATA LEAK DETECTED! Failed to merge: {missing_teams}")
else:
    print("[INFO] All tournament teams successfully matched!")

# 4. Injecting Data into the Tournament Bracket
print("[INFO] Injecting features into upcoming fixtures...")
df_fixtures = pd.merge(df_fixtures, df_profiles, left_on='home_team', right_on='team', how='left')
df_fixtures = df_fixtures.rename(columns={'current_elo': 'home_elo', 'Poss': 'home_poss', 'Gls': 'home_gls'})
df_fixtures = df_fixtures.drop('team', axis=1)

df_fixtures = pd.merge(df_fixtures, df_profiles, left_on='away_team', right_on='team', how='left')
df_fixtures = df_fixtures.rename(columns={'current_elo': 'away_elo', 'Poss': 'away_poss', 'Gls': 'away_gls'})
df_fixtures = df_fixtures.drop('team', axis=1)

# 5. FEATURE ENGINEERING (Differentials)
df_fixtures['elo_diff'] = df_fixtures['home_elo'] - df_fixtures['away_elo']
df_fixtures['poss_diff'] = df_fixtures['home_poss'] - df_fixtures['away_poss']
df_fixtures['gls_diff'] = df_fixtures['home_gls'] - df_fixtures['away_gls']

# Save the AI-ready dataset
df_fixtures.to_csv(OUTPUT_MATCHUPS_PATH, index=False)

print("\n" + "="*50)
print(" FEATURE ENGINEERING SUCCESSFUL")
print("="*50)
print(f"Data Sample (What the AI will see):")
print(df_fixtures[['home_team', 'away_team', 'elo_diff', 'poss_diff', 'gls_diff']].head().to_string(index=False))

[INFO] Loading generated datasets...
[INFO] Built integrated profiles for 47 out of 48 teams.
[INFO] All tournament teams successfully matched!
[INFO] Injecting features into upcoming fixtures...

 FEATURE ENGINEERING SUCCESSFUL
Data Sample (What the AI will see):
    home_team              away_team   elo_diff  poss_diff  gls_diff
      England               DR Congo 240.872781       26.0         2
      Belgium                Senegal  78.948135        2.0        -3
United States Bosnia and Herzegovina 185.356367       16.0         2
        Spain                Austria 203.234077       21.3        -1
     Portugal                Croatia  63.881588        9.0         0


In [ ]:
import pandas as pd
from pathlib import Path

# 1. Flexible Paths Configuration
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

ELO_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"
STATS_PATH = BASE_DIR / "data" / "processed" / "fbref_group_stats.csv"
FIXTURES_PATH = BASE_DIR / "data" / "processed" / "upcoming_fixtures.csv"
OUTPUT_MATCHUPS_PATH = BASE_DIR / "data" / "processed" / "model_input_fixtures.csv"

print("[INFO] Loading generated datasets...")
df_elo = pd.read_csv(ELO_PATH)
df_stats = pd.read_csv(STATS_PATH)
df_fixtures = pd.read_csv(FIXTURES_PATH)

# 2. Bulletproof Name Standardization (Substring Matching)
# Exact matching fails due to invisible characters or weird hyphens. 
# We use partial string detection instead.
def standardize_team_names(name):
    name = str(name).strip()
    if "Bosnia" in name: return "Bosnia and Herzegovina"
    if "USA" in name or name == "US": return "United States"
    if "Korea" in name and "South" not in name and "North" not in name: return "South Korea"
    if "Iran" in name: return "Iran"
    if "Cote" in name or "Côte" in name or "Ivory" in name: return "Ivory Coast"
    if "Cabo Verde" in name: return "Cape Verde"
    if "Congo" in name and "DR" in name: return "DR Congo"
    return name

# Apply the foolproof mapping across all dataframes
for df in [df_stats, df_elo]:
    df['team'] = df['team'].apply(standardize_team_names)

for col in ['home_team', 'away_team']:
    df_fixtures[col] = df_fixtures[col].apply(standardize_team_names)

# 3. Data Blending (Creating Team Profiles)
df_profiles = pd.merge(df_elo, df_stats[['team', 'Poss', 'Gls']], on='team', how='inner')
print(f"[INFO] Built integrated profiles for {len(df_profiles)} out of 48 teams.")

# Anomaly Detection
tournament_teams = set(df_fixtures['home_team']).union(set(df_fixtures['away_team']))
missing_teams = tournament_teams - set(df_profiles['team'])

if missing_teams:
    print(f"\n[WARNING] DATA LEAK DETECTED! Failed to merge: {missing_teams}")
else:
    print("[INFO] All tournament teams successfully matched!")

# 4. Injecting Data into the Tournament Bracket
print("[INFO] Injecting features into upcoming fixtures...")
df_fixtures = pd.merge(df_fixtures, df_profiles, left_on='home_team', right_on='team', how='left')
df_fixtures = df_fixtures.rename(columns={'current_elo': 'home_elo', 'Poss': 'home_poss', 'Gls': 'home_gls'})
df_fixtures = df_fixtures.drop('team', axis=1)

df_fixtures = pd.merge(df_fixtures, df_profiles, left_on='away_team', right_on='team', how='left')
df_fixtures = df_fixtures.rename(columns={'current_elo': 'away_elo', 'Poss': 'away_poss', 'Gls': 'away_gls'})
df_fixtures = df_fixtures.drop('team', axis=1)

# 5. FEATURE ENGINEERING (Differentials)
df_fixtures['elo_diff'] = df_fixtures['home_elo'] - df_fixtures['away_elo']
df_fixtures['poss_diff'] = df_fixtures['home_poss'] - df_fixtures['away_poss']
df_fixtures['gls_diff'] = df_fixtures['home_gls'] - df_fixtures['away_gls']

# Save the AI-ready dataset
df_fixtures.to_csv(OUTPUT_MATCHUPS_PATH, index=False)

print(" FEATURE ENGINEERING SUCCESSFUL")
print("="*50)
print(f"Data Sample (What the AI will see):")
print(df_fixtures[['home_team', 'away_team', 'elo_diff', 'poss_diff', 'gls_diff']].head().to_string(index=False))

[INFO] Loading generated datasets...
[INFO] Built integrated profiles for 47 out of 48 teams.
[INFO] All tournament teams successfully matched!
[INFO] Injecting features into upcoming fixtures...

 FEATURE ENGINEERING SUCCESSFUL
Data Sample (What the AI will see):
    home_team              away_team   elo_diff  poss_diff  gls_diff
      England               DR Congo 240.872781       26.0         2
      Belgium                Senegal  78.948135        2.0        -3
United States Bosnia and Herzegovina 185.356367       16.0         2
        Spain                Austria 203.234077       21.3        -1
     Portugal                Croatia  63.881588        9.0         0
